In [7]:
# %%
# =============================================================================
# Célula 1: Imports e Configurações Globais
# =============================================================================
import pandas as pd
import numpy as np
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
import json
import traceback
import glob # Para encontrar arquivos de log
from sklearn.metrics import accuracy_score, f1_score

# --- Imports da Biblioteca ---
try:
    from activetextclassification.data_preparation import load_split_and_preprocess_data 
    from activetextclassification.models import get_model # BaseTextClassifier, BaseFeatureClassifier não são mais usados diretamente aqui
    from activetextclassification.embeddings import get_embedder # BaseEmbedder
    from activetextclassification.optimization.genetic_l0_optimizer import GeneticL0Optimizer
    from activetextclassification.utils import preprocess_label
    print("Módulos da biblioteca activetextclassification importados.")
except Exception as e: 
    print(f"ERRO ao importar biblioteca: {e}") 
    raise e

# --- Autoreload ---
try: 
    %load_ext autoreload; 
    %autoreload 2; 
    print("Autoreload ativado.")
except Exception: pass

# --- Parâmetros do Experimento de Otimização ---
# <<< MUDE AQUI CONFORME NECESSÁRIO >>>
DATA_FILE = r'../data/dataset.csv'
TEXT_COLUMN = 'nm_item'
LABEL_COLUMN = 'nm_product'
MIN_SAMPLES_PREPROC = 5
RARE_LABEL_PREPROC = "_RARE_"
CLASSIFIER_CONFIG_AG = {
    'type': 'PVBin',
    'params': {'method':'binary', 'query':'binary', 'norm':None, 'query_norm':None, 'ngram_range': [1,2]}
}
GLOBAL_EMBEDDER_CONFIG_AG = None
L0_SIZE_TO_OPTIMIZE = 200 # Teste com tamanho pequeno
POPULATION_SIZE_AG = 20   # População pequena para teste
N_GENERATIONS_AG = 2    # Poucas gerações para teste
CROSSOVER_RATE_AG = 0.8
MUTATION_RATE_AG = 0.2
MUTATION_STRENGTH_AG = 20 # Mutar 1 gene
ELITISM_RATE_AG = 0.10
TOURNAMENT_SIZE_AG = 3
LOG_DETAILED_FITNESS_AG = True # Ativar log detalhado

# Nomes base para arquivos de saída (serão sufixados)
OPTIMIZATION_DIR = f"ag_optimization_results_L0_{L0_SIZE_TO_OPTIMIZE}" # Diretório para organizar
os.makedirs(OPTIMIZATION_DIR, exist_ok=True) # Criar diretório
# Diretório para salvar os splits (pode ser o mesmo OPTIMIZATION_DIR ou um subdiretório)
DATA_SPLIT_DIR = r'../data/data_splits' 
os.makedirs(DATA_SPLIT_DIR, exist_ok=True) # Criar diretório
OPTIMIZATION_HISTORY_BASE_FILE = os.path.join(OPTIMIZATION_DIR, f"ag_history")
BEST_L0_BASE_FILE = os.path.join(OPTIMIZATION_DIR, f"ag_best_l0")
DETAILED_FITNESS_LOG_BASE_FILE = os.path.join(OPTIMIZATION_DIR, f"ag_detailed_fitness")
# <<< FIM MUDE AQUI >>>

print(f"Experimento de Otimização de L0 com AG") # ... (outros prints de config)
print(f"Resultados serão salvos em: {os.path.abspath(OPTIMIZATION_DIR)}")
CHECKPOINT_DIR_AG = os.path.join(OPTIMIZATION_DIR, "checkpoints") # Diretório para checkpoints do AG
CHECKPOINT_PREFIX_AG = "ag_opt" # Prefixo para arquivos de checkpoint
#

Módulos da biblioteca activetextclassification importados.
Experimento de Otimização de L0 com AG
Resultados serão salvos em: d:\Nuvem\ghdaru\OneDrive\030_DOUTORADO\120_TESE\130_TESEGIT\activetextclassification\examples\ag_optimization_results_L0_200


In [8]:
df_train_pool, df_test_eval, all_possible_labels_original_for_f1 = load_split_and_preprocess_data(
    file_path=DATA_FILE,
    text_column=TEXT_COLUMN,
    label_column=LABEL_COLUMN,
    min_samples_per_class=MIN_SAMPLES_PREPROC, # Usa o mesmo parâmetro do notebook
    rare_group_label=RARE_LABEL_PREPROC,       # Usa o mesmo parâmetro
    test_set_size=0.30,                        # Definido aqui ou como parâmetro global
    random_state_split=42,                     # Semente para divisão
    output_dir=DATA_SPLIT_DIR,                 # Onde salvar/carregar os CSVs
    force_split=False                          # Mude para True para refazer a divisão
)

if df_train_pool is None or df_test_eval is None or all_possible_labels_original_for_f1 is None:
    raise SystemExit("Falha crítica na preparação e divisão dos dados. Abortando.")

# df_full_ag agora é df_train_pool
df_full_ag = df_train_pool
# all_possible_labels_ag agora é all_possible_labels_original_for_f1
all_possible_labels_ag = all_possible_labels_original_for_f1

print(f"Pool de Treino/Otimização (df_full_ag) pronto: {len(df_full_ag)} amostras.")
print(f"Conjunto de Teste para Avaliação (df_test_eval) pronto: {len(df_test_eval)} amostras.")
print(f"Total de labels únicos para F1-score: {len(all_possible_labels_ag)}")


# Verificar se o L0_SIZE_TO_OPTIMIZE é válido para df_full_ag
if df_full_ag.empty or L0_SIZE_TO_OPTIMIZE > len(df_full_ag): 
    raise ValueError(f"Dataset de treino/otimização AG inválido (tamanho: {len(df_full_ag)}) ou L0_SIZE_TO_OPTIMIZE ({L0_SIZE_TO_OPTIMIZE}) muito grande.")

# Preparar Embedder Global (treinado APENAS em df_full_ag, que é o df_train_pool)
embedder_instance_ag = None 
# ... (código para treinar embedder_instance_ag como antes, usando df_full_ag) ...
clf_type_ag = CLASSIFIER_CONFIG_AG.get('type')
if clf_type_ag in ['GNB', 'LSVC', 'LR', 'SGD'] and GLOBAL_EMBEDDER_CONFIG_AG:
    print("--- Preparando Embedder Global (treinado em df_full_ag / df_train_pool) ---")
    embedder_instance_ag = get_embedder(GLOBAL_EMBEDDER_CONFIG_AG)
    embedder_instance_ag.fit(df_full_ag[TEXT_COLUMN].tolist(), df_full_ag[LABEL_COLUMN].tolist())
elif clf_type_ag in ['GNB', 'LSVC', 'LR', 'SGD'] and not GLOBAL_EMBEDDER_CONFIG_AG: 
    raise ValueError("Configuração de Embedder global ausente para classificador baseado em features.")

--- Iniciando Carga, Pré-processamento e Divisão de Dados ---
Carregando conjuntos de dados divididos e labels de '../data/data_splits'...
Dados carregados com sucesso.
Pool de Treino/Otimização (df_full_ag) pronto: 175255 amostras.
Conjunto de Teste para Avaliação (df_test_eval) pronto: 75110 amostras.
Total de labels únicos para F1-score: 622


In [9]:
# --- CALCULAR PERFORMANCE BASELINE ---
print("\n--- Calculando Performance Baseline (Treino em D_train_opt, Avaliação em T) ---")

if df_train_pool.empty or df_test_eval.empty:
    print("ERRO: Conjunto de treino ou teste está vazio. Não é possível calcular o baseline.")
else:
    try:
        # Preparar dados para o classificador baseline
        X_train_baseline = df_train_pool[TEXT_COLUMN].tolist()
        y_train_baseline = df_train_pool[LABEL_COLUMN].tolist()
        
        X_test_baseline = df_test_eval[TEXT_COLUMN].tolist()
        y_test_baseline_true = df_test_eval[LABEL_COLUMN].tolist()

        # Instanciar e treinar o modelo baseline (usando CLASSIFIER_CONFIG_AG)
        # Assumindo que PVBin não precisa de embedder global treinado separadamente
        # Se precisasse, o embedder deveria ser treinado APENAS em X_train_baseline
        
        print(f"Configuração do Classificador Baseline: {CLASSIFIER_CONFIG_AG}")
        baseline_model = get_model(CLASSIFIER_CONFIG_AG)
        
        # Treinar o modelo baseline no conjunto de treino/otimização (70%)
        print(f"Treinando modelo baseline em {len(X_train_baseline)} amostras de treino...")
        baseline_model.fit(X_train_baseline, y_train_baseline)
        print("Modelo baseline treinado.")

        # Avaliar no conjunto de teste T (30%)
        print(f"Avaliando modelo baseline em {len(X_test_baseline)} amostras de teste T...")
        y_pred_baseline = baseline_model.predict(X_test_baseline)
        print("Predições no conjunto de teste T concluídas.")

        baseline_accuracy = accuracy_score(y_test_baseline_true, y_pred_baseline)
        baseline_f1_macro = f1_score(y_test_baseline_true, y_pred_baseline, 
                                     average='macro', 
                                     labels=all_possible_labels_original_for_f1, # Usar todos os labels originais
                                     zero_division=0)
        
        print("\n--- RESULTADOS BASELINE (PVBin Treinado em D_train_opt, Avaliado em T) ---")
        print(f"Acurácia Baseline: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
        print(f"F1-Macro Baseline: {baseline_f1_macro:.4f} ({baseline_f1_macro*100:.2f}%)")

        # Guardar os resultados do baseline para referência futura, se necessário
        baseline_results = {
            "classifier_config": CLASSIFIER_CONFIG_AG,
            "train_set_size": len(df_train_pool),
            "test_set_size": len(df_test_eval),
            "accuracy_on_T": baseline_accuracy,
            "f1_macro_on_T": baseline_f1_macro
        }
        baseline_file = os.path.join(OPTIMIZATION_DIR, "baseline_performance_on_T.json")
        with open(baseline_file, 'w') as f:
            json.dump(baseline_results, f, indent=4)
        print(f"Resultados baseline salvos em: {baseline_file}")

    except Exception as e_baseline:
        print(f"ERRO ao calcular performance baseline: {e_baseline}")
        import traceback
        traceback.print_exc()

# df_full_ag e all_possible_labels_ag são preparados para as células seguintes do AG
df_full_ag = df_train_pool
all_possible_labels_ag = all_possible_labels_original_for_f1

# Preparar Embedder Global (se necessário, treinado APENAS em df_full_ag (df_train_pool))
# Esta parte é para o AG, não para o baseline do PVBin que já foi calculado.
embedder_instance_ag = None 
clf_type_ag_for_embedder = CLASSIFIER_CONFIG_AG.get('type') # Re-checar config para o AG
if clf_type_ag_for_embedder in ['GNB', 'LSVC', 'LR', 'SGD'] and GLOBAL_EMBEDDER_CONFIG_AG:
    print("--- Preparando Embedder Global para o AG (treinado em df_full_ag) ---")
    embedder_instance_ag = get_embedder(GLOBAL_EMBEDDER_CONFIG_AG)
    embedder_instance_ag.fit(df_full_ag[TEXT_COLUMN].tolist(), df_full_ag[LABEL_COLUMN].tolist())
elif clf_type_ag_for_embedder in ['GNB', 'LSVC', 'LR', 'SGD'] and not GLOBAL_EMBEDDER_CONFIG_AG: 
    print("AVISO: Configuração de Embedder global ausente para classificador do AG baseado em features.")
    # raise ValueError("Configuração de Embedder global ausente para classificador do AG baseado em features.") # Descomente se for crítico


--- Calculando Performance Baseline (Treino em D_train_opt, Avaliação em T) ---
Configuração do Classificador Baseline: {'type': 'PVBin', 'params': {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}}
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Treinando modelo baseline em 175255 amostras de treino...
Fitting ProductVectorizerClassifier com 175255 amostras...
Fit concluído.
Modelo baseline treinado.
Avaliando modelo baseline em 75110 amostras de teste T...
Predições no conjunto de teste T concluídas.

--- RESULTADOS BASELINE (PVBin Treinado em D_train_opt, Avaliado em T) ---
Acurácia Baseline: 0.8884 (88.84%)
F1-Macro Baseline: 0.7160 (71.60%)
Resultados baseline salvos em: ag_optimization_results_L0_200\baseline_performance_on_T.json


In [10]:
# CÉLULA 3: Função run_optimization_with_checkpointing MODIFICADA

def run_optimization_with_checkpointing(
    df_l0_pool,                
    df_evaluation,             
    optimization_goal, 
    fitness_metric_to_optimize, 
    current_n_generations_config, # << NOVO PARÂMETRO: N_GENERATIONS_AG atual
    seed_offset=0, 
    force_restart_ag=False 
):
    goal_suffix = f"_{fitness_metric_to_optimize.split('_')[0].upper()}_{optimization_goal.upper()}"
    file_hist_final = f"{OPTIMIZATION_HISTORY_BASE_FILE}{goal_suffix}.xlsx"
    file_best_l0_final = f"{BEST_L0_BASE_FILE}{goal_suffix}.csv"
    detailed_log_file_path = f"{DETAILED_FITNESS_LOG_BASE_FILE}{fitness_metric_to_optimize.split('_')[0].upper()}_{optimization_goal.upper()}.csv"

    # Verificar se os arquivos FINAIS já existem
    final_results_exist = os.path.exists(file_hist_final) and os.path.exists(file_best_l0_final)
    
    if final_results_exist and not force_restart_ag:
        try:
            # Tentar ler o histórico para ver quantas gerações foram rodadas
            history_df_existing = pd.read_excel(file_hist_final)
            generations_in_history = 0
            if not history_df_existing.empty and 'generation' in history_df_existing.columns:
                generations_in_history = history_df_existing['generation'].max()
            
            if generations_in_history >= current_n_generations_config:
                print(f"\n{'='*10} Otimização para {goal_suffix} JÁ TEM RESULTADOS FINAIS com {generations_in_history} ger. (>= {current_n_generations_config} config.). Pulando. {'='*10}")
                best_l0_df_saved = pd.read_csv(file_best_l0_final)
                best_perf_saved = np.nan
                if not best_l0_df_saved.empty: # Checar se a coluna existe antes de acessá-la
                    if 'metric_value_on_eval_set' in best_l0_df_saved.columns:
                         best_perf_saved = best_l0_df_saved['metric_value_on_eval_set'].iloc[0] 
                    elif 'metric_value' in best_l0_df_saved.columns: # Fallback para nome antigo
                         best_perf_saved = best_l0_df_saved['metric_value'].iloc[0]

                return None, best_perf_saved, history_df_existing
            else:
                print(f"\n--- Resultados finais existem para {goal_suffix} com {generations_in_history} ger., mas N_GENERATIONS_AG atual é {current_n_generations_config}. Tentando continuar/resumir. ---")
        except Exception as e_load_final:
            print(f"AVISO: Falha ao verificar histórico existente para {goal_suffix}: {e_load_final}. Tentando continuar/resumir.")
            # Se não puder ler o histórico, prossegue para rodar/resumir
    
    # Se force_restart_ag é True, ou se os resultados finais não existem, ou se queremos mais gerações:
    print(f"\n{'='*15} Iniciando/Resumindo Otimização (L0Tam: {L0_SIZE_TO_OPTIMIZE}, Obj: {optimization_goal.upper()}, Métrica: {fitness_metric_to_optimize}, Total Ger: {current_n_generations_config}) {'='*15}")

    current_checkpoint_prefix = f"{CHECKPOINT_PREFIX_AG}_{goal_suffix}"
    checkpoint_file_to_check = os.path.join(CHECKPOINT_DIR_AG, f"{current_checkpoint_prefix}_l0_{L0_SIZE_TO_OPTIMIZE}_pop_{POPULATION_SIZE_AG}_gen_{current_n_generations_config}_{fitness_metric_to_optimize.replace('_on_full','')}_{optimization_goal}_ckpt.pkl")
    
    # ATENÇÃO: O nome do checkpoint inclui N_GENERATIONS. 
    # Se você muda N_GENERATIONS_AG no notebook, o nome do checkpoint procurado/salvo também muda.
    # Para resumir, o N_GENERATIONS no nome do checkpoint DEVE ser o N_GENERATIONS da execução que você quer resumir.
    # Ou, o N_GENERATIONS não deve fazer parte do nome do checkpoint se você quer que ele seja flexível.
    # VAMOS REMOVER N_GENERATIONS_AG DO NOME DO CHECKPOINT PARA FACILITAR O RESUMO COM N_GENERATIONS VARIÁVEL.
    
    checkpoint_file_flexible_name = os.path.join(
        CHECKPOINT_DIR_AG, 
        f"{current_checkpoint_prefix}_l0_{L0_SIZE_TO_OPTIMIZE}_pop_{POPULATION_SIZE_AG}_{fitness_metric_to_optimize.replace('_on_full','')}_{optimization_goal}_ckpt.pkl"
    )

    if force_restart_ag and os.path.exists(checkpoint_file_flexible_name):
        try:
            os.remove(checkpoint_file_flexible_name)
            print(f"   Checkpoint antigo {checkpoint_file_flexible_name} removido devido a force_restart_ag=True.")
        except Exception as e_rm_ckpt:
            print(f"   AVISO: Não foi possível remover o checkpoint antigo {checkpoint_file_flexible_name}: {e_rm_ckpt}")

    ag_optimizer = GeneticL0Optimizer(
        df_full=df_l0_pool,             
        df_evaluation_set=df_evaluation, 
        text_column=TEXT_COLUMN, 
        label_column=LABEL_COLUMN,
        classifier_config=CLASSIFIER_CONFIG_AG, 
        initial_l0_size=L0_SIZE_TO_OPTIMIZE,
        all_possible_labels=all_possible_labels_ag, 
        population_size=POPULATION_SIZE_AG,
        n_generations=current_n_generations_config, # << PASSA O N_GENERATIONS ATUAL
        crossover_rate=CROSSOVER_RATE_AG,
        mutation_rate=MUTATION_RATE_AG, 
        mutation_strength=MUTATION_STRENGTH_AG,
        elitism_rate=ELITISM_RATE_AG, 
        fitness_metric=fitness_metric_to_optimize, 
        optimization_goal=optimization_goal, 
        tournament_size=TOURNAMENT_SIZE_AG,
        random_seed=42 + seed_offset, 
        embedder=embedder_instance_ag,
        log_detailed_fitness=LOG_DETAILED_FITNESS_AG,
        checkpoint_dir=CHECKPOINT_DIR_AG, 
        checkpoint_prefix=current_checkpoint_prefix 
    )
    
    # Modificar o AG para que o checkpoint_file não dependa de n_generations
    # Isso foi feito no construtor do AG na minha sugestão anterior.
    # Se o AG já faz isso, ótimo. Se não, o AG precisa ser ajustado ou o nome do checkpoint aqui.
    ag_optimizer.checkpoint_file = checkpoint_file_flexible_name # Sobrescrever para garantir o nome flexível

    best_l0_indices, best_actual_perf, history_df = ag_optimizer.run_optimization(
        detailed_log_file_path_from_notebook=detailed_log_file_path
    )

    if history_df is not None and not history_df.empty:
        print(f"Salvando histórico FINAL ({goal_suffix}) em: {file_hist_final}")
        try: history_df.to_excel(file_hist_final, index=False)
        except Exception as e: print(f"Erro salvar histórico FINAL AG ({goal_suffix}): {e}")

    if best_l0_indices is not None and len(best_l0_indices) > 0:
        print(f"Salvando L0 FINAL (Obj: {optimization_goal}, Perf no EvalSet: {best_actual_perf:.4f}) em: {file_best_l0_final}")
        best_l0_df = df_l0_pool.iloc[best_l0_indices][[TEXT_COLUMN, LABEL_COLUMN]].copy()
        best_l0_df['metric_value_on_eval_set'] = best_actual_perf 
        best_l0_df['metric_type'] = fitness_metric_to_optimize # Adicionado para consistência
        best_l0_df['optimization_goal'] = optimization_goal # Adicionado
        # best_l0_df['l0_indices_str'] = ",".join(map(str, best_l0_indices)) # Adicionado
        try: best_l0_df.to_csv(file_best_l0_final, index=False, encoding='utf-8-sig')
        except Exception as e: print(f"Erro salvar L0 FINAL ({goal_suffix}): {e}")
    
    return best_l0_indices, best_actual_perf, history_df

In [11]:
# %%
# =============================================================================
# Célula 4: Executar Otimizações
# =============================================================================
FORCE_RESTART_ALL_AG = False # Mude para True para ignorar checkpoints e refazer do zero.
optimization_results = {} # <--- INICIALIZAÇÃO ADICIONADA AQUI

if df_full_ag is not None and not df_full_ag.empty and df_test_eval is not None and not df_test_eval.empty:
    # --- Otimizar para ACURÁCIA ---
    res_acc_max = run_optimization_with_checkpointing( # Nome da função atualizado
        df_l0_pool=df_full_ag, df_evaluation=df_test_eval,    
        optimization_goal='maximize', fitness_metric_to_optimize='accuracy_on_full',
        current_n_generations_config=N_GENERATIONS_AG, # << PASSA N_GENERATIONS_AG ATUAL
        seed_offset=0, force_restart_ag=FORCE_RESTART_ALL_AG 
    )
    # Adicionar checagem se res_acc_max não é None (caso a otimização seja pulada e retorne None)
    if res_acc_max is not None:
        optimization_results['ACC_MAX'] = {'indices': res_acc_max[0], 'performance': res_acc_max[1], 'history_df': res_acc_max[2]}
    else:
        optimization_results['ACC_MAX'] = {'indices': None, 'performance': np.nan, 'history_df': pd.DataFrame()}


    res_acc_min = run_optimization_with_checkpointing(
        df_l0_pool=df_full_ag, df_evaluation=df_test_eval,    
        optimization_goal='minimize', fitness_metric_to_optimize='accuracy_on_full',
        current_n_generations_config=N_GENERATIONS_AG, # << PASSA N_GENERATIONS_AG ATUAL
        seed_offset=100, force_restart_ag=FORCE_RESTART_ALL_AG
    )
    if res_acc_min is not None:
        optimization_results['ACC_MIN'] = {'indices': res_acc_min[0], 'performance': res_acc_min[1], 'history_df': res_acc_min[2]}
    else:
        optimization_results['ACC_MIN'] = {'indices': None, 'performance': np.nan, 'history_df': pd.DataFrame()}


    # --- Otimizar para F1-SCORE ---
    res_f1_max = run_optimization_with_checkpointing(
        df_l0_pool=df_full_ag, df_evaluation=df_test_eval,    
        optimization_goal='maximize', fitness_metric_to_optimize='f1_macro_on_full',
        current_n_generations_config=N_GENERATIONS_AG, # << PASSA N_GENERATIONS_AG ATUAL
        seed_offset=200, force_restart_ag=FORCE_RESTART_ALL_AG
    )
    if res_f1_max is not None:
        optimization_results['F1_MAX'] = {'indices': res_f1_max[0], 'performance': res_f1_max[1], 'history_df': res_f1_max[2]}
    else:
        optimization_results['F1_MAX'] = {'indices': None, 'performance': np.nan, 'history_df': pd.DataFrame()}


    res_f1_min = run_optimization_with_checkpointing(
        df_l0_pool=df_full_ag, df_evaluation=df_test_eval,    
        optimization_goal='minimize', fitness_metric_to_optimize='f1_macro_on_full',
        current_n_generations_config=N_GENERATIONS_AG, # << PASSA N_GENERATIONS_AG ATUAL
        seed_offset=300, force_restart_ag=FORCE_RESTART_ALL_AG
    )
    if res_f1_min is not None:
        optimization_results['F1_MIN'] = {'indices': res_f1_min[0], 'performance': res_f1_min[1], 'history_df': res_f1_min[2]}
    else:
        optimization_results['F1_MIN'] = {'indices': None, 'performance': np.nan, 'history_df': pd.DataFrame()}


    print("\n--- Todas as Otimizações Programadas Concluídas/Resumidas/Puladas ---")
else:
    print("ERRO: df_full_ag (pool de treino) ou df_test_eval (conjunto de avaliação) não estão definidos ou estão vazios.")
    # Se os dataframes não estiverem prontos, inicializar optimization_results para evitar erros em células posteriores
    keys = ['ACC_MAX', 'ACC_MIN', 'F1_MAX', 'F1_MIN']
    for key in keys:
        if key not in optimization_results:
             optimization_results[key] = {'indices': None, 'performance': np.nan, 'history_df': pd.DataFrame()}


--- Resultados finais existem para _ACCURACY_MAXIMIZE com 1 ger., mas N_GENERATIONS_AG atual é 2. Tentando continuar/resumir. ---

=============== Iniciando/Resumindo Otimização (L0Tam: 200, Obj: MAXIMIZE, Métrica: accuracy_on_full, Total Ger: 2) ===============
GeneticL0Optimizer inicializado.
 - Pool para L0s (df_full): 175255 amostras.
 - Avaliação Fitness em (df_eval): 75110 amostras.
 - População: 20, Gerações: 2, L0 Size: 200
 - Objetivo: maximize, Métrica: accuracy_on_full (avaliada em df_eval)
 - Log Detalhado será ativado.
 - Checkpoint será salvo em/carregado de: ag_optimization_results_L0_200\checkpoints\ag_opt__ACCURACY_MAXIMIZE_l0_200_pop_20_gen_2_accuracy_maximize_ckpt.pkl
   Checkpoint carregado. Resumindo da geração 1 (próxima a ser executada).
   Melhor performance até agora: 0.3617
   Continuando log detalhado em: ag_optimization_results_L0_200\ag_detailed_fitnessACCURACY_MAXIMIZE.csv


AG (Melhor: 0.3617):   0%|          | 0/2 [00:00<?, ?it/s]

Fitness Gen 1:   0%|          | 0/20 [00:00<?, ?it/s]

Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 200 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 200 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 200 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1

Fitness Gen 2:   0%|          | 0/20 [00:00<?, ?it/s]

...

AG (Melhor: 0.0564):   0%|          | 0/2 [00:00<?, ?it/s]

Fitness Gen 1:   0%|          | 0/20 [00:00<?, ?it/s]

Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 200 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 200 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 200 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1

Fitness Gen 2:   0%|          | 0/20 [00:00<?, ?it/s]

Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 200 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 200 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 200 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1